# 06 · Baseline race model

Two parts, matching how the race simulator will work later:

1. **DNF model**: chance a driver doesn't take the chequered flag (retired, disqualified, didn't start). Lapped cars that finish count as finishers. Benchmark: the training-set DNF rate for everyone.
2. **Position model**: finishing order among the drivers who finished. Benchmark: finish in grid order.

Each is fitted as a **linear model** and a **gradient-boosted tree model**, trained on 2018–2024 and tested on 2025.

**Inputs**, all using only information available before the race:

| Input | Meaning |
|---|---|
| Grid, PitLaneStart | starting slot; a pit-lane start counts as last |
| QualiGapPct | this weekend's qualifying gap to teammate (%) |
| DriverForm, TeamForm | mean finishing rank in their last 5 finishes |
| RacePaceGapPct | mean race-pace gap to teammate over the last 5 dry races |
| DriverDNFRate, TeamDNFRate | share of DNFs in the last 10 races |
| CircuitPosChange | how much the order usually shuffles at this circuit (earlier editions) |
| CircuitDNFRate | DNF rate at this circuit in earlier editions |

**Limitations:** team form carries over between seasons even when the car changes a lot, and it restarts when a team rebrands (e.g. Racing Point → Aston Martin).

Run notebooks 00, 03 and 05 first.

In [1]:
# @title Setup: mount Drive, get the latest code, install FastF1
REPO_URL = "https://github.com/khushi075/f1-race-sim.git"  # @param {type:"string"}
DATA_DIR = "/content/drive/MyDrive/f1sim_data"  # @param {type:"string"}

import os, sys
from google.colab import drive

try:
    import imp
except ImportError:
    import types, importlib
    imp = types.ModuleType('imp')
    imp.reload = importlib.reload
    sys.modules['imp'] = imp

drive.mount("/content/drive")

REPO_DIR = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q "fastf1>=3.9,<4"

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.environ["F1SIM_DATA_DIR"] = DATA_DIR

%load_ext autoreload
%autoreload 2
print("Code:", REPO_DIR, "| Data:", DATA_DIR)

Mounted at /content/drive
Cloning into '/content/f1-race-sim'...
remote: Enumerating objects: 135, done.
remote: Counting objects: 100% (135/135), done.
remote: Compressing objects: 100% (101/101), done.
remote: Total 135 (delta 74), reused 81 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (135/135), 89.84 KiB | 4.73 MiB/s, done.
Resolving deltas: 100% (74/74), done.
/content/f1-race-sim
Code: /content/f1-race-sim | Data: /content/drive/MyDrive/f1sim_data


In [2]:
# @title Build the dataset
TEST_YEAR = 2025  # @param {type:"integer"}
WET_RULE = "tyres"  # @param ["tyres", "tyres_or_rain"]

import pandas as pd
import plotly.express as px
from f1sim.paths import get_paths
from f1sim.ingest import load_quali
from f1sim.ingest_results import load_results
from f1sim.ingest_tracing import load_race_ti
from f1sim.features import teammate_gaps
from f1sim.race_pace import clean_dry_laps, teammate_race_gaps
from f1sim.baseline import (build_dataset, fit_evaluate, is_finished,
                            POSITION_FEATURES, DNF_FEATURES)

paths = get_paths()
results = load_results(paths)
quali_gaps = teammate_gaps(load_quali(paths))
race_gaps, _ = teammate_race_gaps(clean_dry_laps(load_race_ti("laps", paths), wet_rule=WET_RULE))
df = build_dataset(results, quali_gaps, race_gaps)

print(f"{len(df):,} driver-race rows | {df['RaceIdx'].nunique()} races | "
      f"{100 * df['Finished'].mean():.1f}% finished")
print("Missing values per input (%):")
(100 * df[sorted(set(POSITION_FEATURES + DNF_FEATURES))].isna().mean()).round(1)

3,458 driver-race rows | 173 races | 85.1% finished
Missing values per input (%):


,0
CircuitDNFRate,0.6
CircuitPosChange,0.6
DriverDNFRate,1.2
DriverForm,1.5
Grid,0.0
GridxCircuitPosChange,0.6
PitLaneStart,0.0
QualiGapPct,6.8
RacePaceGapPct,1.7
TeamDNFRate,0.9


In [3]:
# @title Check the DNF rule: how each status was classified
status = results["Status"].value_counts().rename("Count").to_frame()
status["CountsAs"] = ["finished" if is_finished(s) else "DNF" for s in status.index]
status

,Count,CountsAs
Status,,
Finished,1995,finished
+1 Lap,536,finished
Lapped,297,finished
Retired,161,DNF
+2 Laps,101,finished
Collision,77,DNF
Accident,35,DNF
Engine,31,DNF
Collision damage,29,DNF


In [4]:
# @title Fit on earlier seasons, score on the test season
out = fit_evaluate(df, test_year=TEST_YEAR)
print(f"Train rows: {out['n_train_rows']:,} | test rows: {out['n_test_rows']:,}")
print("\nDNF model (lower LogLoss/Brier is better; AUC 0.5 = no better than chance):")
display(out["dnf_metrics"])
print("Position model, finishers only (higher Spearman, lower MAE is better):")
display(out["position_metrics"])
if len(out["position_unscored"]):
    print("Left out of the position scores (no grid position):")
    display(out["position_unscored"])

Train rows: 2,979 | test rows: 479

DNF model (lower LogLoss/Brier is better; AUC 0.5 = no better than chance):


,LogLoss,Brier,AUC
Model,,,
Base rate,0.3804,0.1103,0.5000
Linear (logistic),0.3783,0.1100,0.5593
Gradient boosting,0.3906,0.1147,0.5453


Position model, finishers only (higher Spearman, lower MAE is better):


,MeanSpearman,MAE_positions,ExactPct,DriversScored
Model,,,,
Grid order,0.755,2.339,24.344,419
Linear (ridge),0.791,2.234,21.241,419
Gradient boosting,0.776,2.344,20.048,419


In [5]:
# @title Race by race: how well each model ordered the finishers
per_race = out["position_per_race"].melt(id_vars=["Round", "Event"], var_name="Model",
                                         value_name="Spearman")
fig = px.line(per_race, x="Round", y="Spearman", color="Model", markers=True,
              hover_data=["Event"],
              title=f"{TEST_YEAR}: rank correlation between predicted and actual finishing order")
fig.show()

In [6]:
# @title What drives the position predictions
coefs = out["position_linear_coefs"].rename("Coefficient").rename_axis("Input").reset_index()
px.bar(coefs, x="Coefficient", y="Input", orientation="h",
       title="Linear model: standardised coefficients (positive = finishes further back)"
       ).update_layout(yaxis={"categoryorder": "total ascending"}).show()

imp = out["position_gb_importance"].rename("Importance").rename_axis("Input").reset_index()
px.bar(imp, x="Importance", y="Input", orientation="h",
       title="Gradient boosting: permutation importance on the test season (positions of MAE lost)"
       ).update_layout(yaxis={"categoryorder": "total ascending"}).show()

In [7]:
# @title One race: predicted vs actual finishing order
ROUND = 1  # @param {type:"integer"}

race = out["position_test_pred"]
race = race[race["Round"] == ROUND].sort_values("FinishRank")
print(race["Event"].iloc[0] if len(race) else f"No finishers found for round {ROUND}")
race.drop(columns=["Year", "Round", "Event"])

Australian Grand Prix


,Abbreviation,TeamName,Grid,FinishRank,Pred (Grid order),Pred (Linear (ridge)),Pred (Gradient boosting)
2979,NOR,McLaren,1.0,1.0,1,1,1
2980,VER,Red Bull,3.0,2.0,3,2,2
2981,RUS,Mercedes,4.0,3.0,4,3,5
2982,ANT,Mercedes,16.0,4.0,11,10,9
2983,ALB,Williams,6.0,5.0,6,9,10
2984,STR,Aston Martin,13.0,6.0,10,11,11
2985,HUL,Sauber,17.0,7.0,12,13,13
2986,LEC,Ferrari,7.0,8.0,7,4,4
2987,PIA,McLaren,2.0,9.0,2,5,3
2988,HAM,Ferrari,8.0,10.0,8,6,6


In [8]:
# @title DNF model: highest predicted DNF chances in the test season
pred = out["dnf_test_pred"]
col = "P_DNF (Linear (logistic))"
pred.sort_values(col, ascending=False).head(20)[
    ["Round", "Event", "Abbreviation", "TeamName", "Grid", col, "DNF", "Status"]]

,Round,Event,Abbreviation,TeamName,Grid,P_DNF (Linear (logistic)),DNF,Status
3038,3,Japanese Grand Prix,STR,Aston Martin,20.0,0.220407,0,Lapped
2992,1,Australian Grand Prix,BEA,Haas F1 Team,20.0,0.220142,0,Finished
3108,7,Emilia Romagna Grand Prix,TSU,Red Bull,20.0,0.219748,0,Finished
3048,4,Bahrain Grand Prix,BEA,Haas F1 Team,20.0,0.219399,0,Finished
3055,4,Bahrain Grand Prix,STR,Aston Martin,19.0,0.212895,0,Finished
3287,16,Italian Grand Prix,HAD,RB F1 Team,19.0,0.212817,0,Finished
3151,9,Spanish Grand Prix,TSU,Red Bull,19.0,0.212094,0,Finished
3172,10,Canadian Grand Prix,GAS,Alpine F1 Team,20.0,0.211073,0,Lapped
3353,19,United States Grand Prix,HAD,RB F1 Team,20.0,0.210831,0,Finished
3293,16,Italian Grand Prix,GAS,Alpine F1 Team,20.0,0.210656,0,Lapped
